# AI Video Generation API - Colab
Run Wan2.1-T2V-1.3B on free T4 GPU, exposes Gradio API for local app

## First: Add HF_TOKEN to Colab Secrets
1. Click 🔑 **Secrets** (left panel)
2. Add `HF_TOKEN` = your token from https://huggingface.co/settings/tokens
3. **Runtime > Change runtime type > T4 GPU**
4. **Runtime > Run all**

In [ ]:
import torch, psutil, os, gc
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    free, total = psutil.virtual_memory()[1:3]
    print(f"System RAM: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
else:
    raise RuntimeError("No GPU! Runtime > Change runtime type > T4 GPU")
print("GPU check passed!")


In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q gradio diffusers transformers accelerate sentencepiece protobuf imageio[ffmpeg] psutil
print("Installed!")


In [ ]:
import torch, gradio as gr, os, time, random
from diffusers import DiffusionPipeline
from diffusers.utils import export_to_video

MODEL = "Wan-AI/Wan2.1-T2V-1.3B"
print(f"Loading {MODEL} (8.2GB VRAM - fits T4)...")

pipe = DiffusionPipeline.from_pretrained(
    MODEL,
    torch_dtype=torch.bfloat16,
    device_map="cuda"
)
print("Model loaded on GPU!")


In [ ]:
def generate(prompt, steps=25, guidance=5.0, seed=-1):
    if seed < 0:
        seed = random.randint(0, 2**31)
    gen = torch.Generator(device="cpu").manual_seed(seed)
    print(f"Generating: {prompt}")
    t0 = time.time()
    result = pipe(
        prompt=prompt,
        num_inference_steps=steps,
        guidance_scale=guidance,
        generator=gen,
        num_frames=81
    ).frames[0]
    elapsed = time.time() - t0
    print(f"Done in {elapsed:.1f}s")
    path = export_to_video(result, fps=16)
    return path

iface = gr.Interface(
    fn=generate,
    inputs=[
        gr.Textbox(label="Prompt", value="A tiger walking through tall grass at sunrise, cinematic"),
        gr.Slider(5, 50, 25, label="Steps"),
        gr.Slider(1.0, 10.0, 5.0, label="Guidance"),
        gr.Number(-1, label="Seed (-1=random)", precision=0),
    ],
    outputs=gr.Video(label="Video"),
    title="AI Video Generator (Wan2.1)",
)

print("="*50)
print("COPY THIS GRADIO LINK (xxx.gradio.app)")
print("Set it as T2V_API_URL in your .env file")
print("="*50)
iface.launch(share=True)
